## Import Libraries

In [ ]:
import torch
from datasets import load_dataset, load_from_disk, DatasetDict
from transformers import LEDTokenizer, LEDForConditionalGeneration
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

import os

## 2. Load Dataset

### 2.1 Conenct google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2.2 Load dataset from drive or download and save it in drive

In [ ]:
dataset = load_dataset("ccdv/arxiv-summarization", "document")
print(dataset)
print(dataset["train"][0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

document/train-00000-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

document/train-00001-of-00015.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

document/train-00002-of-00015.parquet:   0%|          | 0.00/226M [00:00<?, ?B/s]

...

document/test-00000-of-00001.parquet:   0%|          | 0.00/104M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})


In [ ]:
dataset_path = '/content/drive/MyDrive/Engineering-UOR/Semester7/Advanced AI/Project/dataset'

# Try loading the dataset
try:
    dataset = load_from_disk(dataset_path)
    print(f"Dataset loaded successfully from {dataset_path}")
except (FileNotFoundError, ValueError) as e:
    print(f"Could not load dataset from {dataset_path}: {e}")
    print("Downloading dataset from Hugging Face...")

    # Download and save
    dataset = load_dataset("ccdv/arxiv-summarization", "document")
    dataset.save_to_disk(dataset_path)
    print(f"Dataset downloaded and saved to {dataset_path}")

# Use the dataset
print(dataset)
print(dataset["train"][0].keys())

Dataset loaded successfully from /content/drive/MyDrive/Engineering-UOR/Semester7/Advanced AI/Project/dataset
DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 203037
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6436
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6440
    })
})
dict_keys(['article', 'abstract'])


### 2.3 Take train and validation dataset



In [ ]:
print("Train Dataset length: ", len(dataset["train"]))
print("Train Dataset length: ", len(dataset["validation"]))

Train Dataset length:  203037
Train Dataset length:  6436


In [ ]:
train_data = dataset["train"].select(range(1000))        # few-shot subset
val_data   = dataset["validation"].select(range(100))   # validation subset

In [ ]:
sample_data_article = train_data[0]["article"]
sample_data_abstract = train_data[0]["abstract"]

print("ARTICLE Lenght:", len(sample_data_article))
print("ABSTRACT Length:", len(sample_data_abstract))

print("\nARTICLE:", sample_data_article)
print("\nABSTRACT:", sample_data_abstract)

ARTICLE Lenght: 26092
ABSTRACT Length: 930

ARTICLE: additive models @xcite provide an important family of models for semiparametric regression or classification . some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability when compared to fully nonparametric models . it is well - known that good estimators in additive models are in general less prone to the curse of high dimensionality than good estimators in fully nonparametric models . many examples of such estimators belong to the large class of regularized kernel based methods over a reproducing kernel hilbert space @xmath0 , see e.g. @xcite . in the last years many interesting results on learning rates of regularized kernel based models for additive models have been published when the focus is on sparsity and when the classical least squares loss function is used , see e.g. @xcite , @xcite , @xcite , @xcite , @xcite , @x

In [ ]:
max_article_len = 0
max_abstract_len = 0
for data in train_data:
    article_len = len(data["article"])
    abstract_len = len(data["abstract"])
    max_article_len = max(max_article_len, article_len)
    max_abstract_len = max(max_abstract_len, abstract_len)

print("Max Article Length:", max_article_len)
print("Max Abstract Length:", max_abstract_len)

Max Article Length: 657995
Max Abstract Length: 68299


### Load model

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("allenai/led-base-16384")
model = AutoModelForSeq2SeqLM.from_pretrained("allenai/led-base-16384")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/648M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [ ]:
def get_token_length(example):
    return {"article_token_len": len(tokenizer(example["article"], truncation=False)["input_ids"]),
            "abstract_token_len": len(tokenizer(example["abstract"], truncation=False)["input_ids"])}

token_lens = train_data.map(get_token_length, remove_columns=train_data.column_names)

print("Max article tokens:", max(token_lens["article_token_len"]))
print("Max abstract tokens:", max(token_lens["abstract_token_len"]))
print("Avg article tokens:", sum(token_lens["article_token_len"]) / len(token_lens))
print("Avg abstract tokens:", sum(token_lens["abstract_token_len"]) / len(token_lens))

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (20399 > 16384). Running this sequence through the model will result in indexing errors


Max article tokens: 244922
Max abstract tokens: 20889
Avg article tokens: 8583.723
Avg article tokens: 371.14465


### Text preprocessing

### PREPROCESS FUNCTION (Prompt Engineering)

In [ ]:
max_input_len = 4096        # LED supports up to 16384, but start smaller
max_target_len = 512        # you can increase for longer summaries

def preprocess(batch):
    inputs = ["summarize: " + art for art in batch["article"]]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_len,
        padding="max_length",
        truncation=True
    )
    # LED requires global attention on <s> token
    model_inputs["global_attention_mask"] = [
        [1] + [0] * (len(input_ids)-1) for input_ids in model_inputs["input_ids"]
    ]

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch["abstract"],
            max_length=max_target_len,
            padding="max_length",
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess, batched=True, remove_columns=val_data.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


In [ ]:
batch_size = 1

training_args = TrainingArguments(
    output_dir="./led_science_summarizer",
    save_total_limit=1,
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/tmp/ipython-input-1846427416.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sjanugopanstudy (sjanugopanstudy-jaanu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,1.854300


TrainOutput(global_step=750, training_loss=1.557543192545573, metrics={'train_runtime': 12646.8998, 'train_samples_per_second': 0.237, 'train_steps_per_second': 0.059, 'total_flos': 3.2402458017792e+16, 'train_loss': 1.557543192545573, 'epoch': 3.0})

In [ ]:
test_article = val_data[0]["article"]
inputs = tokenizer("summarize: " + test_article, return_tensors="pt", max_length=4096, truncation=True).to(model.device)
output = model.generate(**inputs, max_length=512, num_beams=4, early_stopping=True)
generated_summary = tokenizer.decode(output[0], skip_special_tokens=True)

print("\n📝 GENERATED ABSTRACT:\n", generated_summary)
print("\n📌 ORIGINAL ABSTRACT:\n", val_data[0]["abstract"])


📝 GENERATED ABSTRACT:
 we study the phase behavior of a nematic liquid crystal confined between a chemically patterned sinusoidal surface and a flat substrate . 
 we find remarkably good agreement between the phase diagrams of various systems calculated using this effective free energy function on the one hand and the original free energy functional on the other hand . 
 we find remarkably good agreement between the phase diagrams of various systems calculated using this effective free energy function and an average surface director orientation of the patterned substrate and obtain an effective free energy for the nematic liquid crystal cell under consideration .

📌 ORIGINAL ABSTRACT:
 we study the phase behavior of a nematic liquid crystal confined between a flat substrate with strong anchoring and a patterned substrate whose structure and local anchoring strength we vary . by first evaluating an effective surface free energy function characterizing the patterned substrate we derive 

### Evaluate the model

In [ ]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.3 MB/s eta 0:00:00


In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=37217dcc68d348bfef30687e4002ec79cb8fbe65bd4f6dd3201b6a8307d1a6ae
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from evaluate import load

# Load metrics
rouge = load("rouge")
bleu = load("bleu")

# Take a few validation samples to evaluate
val_texts = val_data.select(range(2))
pred_summaries = []
ref_summaries = []

# Generate predictions and collect references
for item in val_texts:
    input_text = "summarize: " + item["article"]
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_length=512,
        num_beams=4,
        early_stopping=True
    )
    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    pred_summaries.append(pred)
    ref_summaries.append(item["abstract"])

# Compute metrics
rouge_scores = rouge.compute(predictions=pred_summaries, references=ref_summaries)
bleu_score = bleu.compute(predictions=pred_summaries, references=[[ref] for ref in ref_summaries])

print("🔎 ROUGE scores:", rouge_scores)
print("🔵 BLEU score:", bleu_score)

🔎 ROUGE scores: {'rouge1': np.float64(0.4718973101608135), 'rouge2': np.float64(0.23633993819838026), 'rougeL': np.float64(0.31277132761231696), 'rougeLsum': np.float64(0.4074421287949809)}
🔵 BLEU score: {'bleu': 0.11739194450938313, 'precisions': [0.6495327102803738, 0.330188679245283, 0.19523809523809524, 0.10096153846153846], 'brevity_penalty': 0.4603809709355699, 'length_ratio': 0.5631578947368421, 'translation_length': 214, 'reference_length': 380}


### Save Model

In [ ]:
model_dir = "/content/drive/MyDrive/Engineering-UOR/Semester7/Advanced AI/Project/model"

os.makedirs(model_dir, exist_ok=True)
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)
print("✅ Model and tokenizer saved to Drive!")

### Reloading your saved model + tokenizer

In [ ]:
model_dir = "/content/drive/MyDrive/Engineering-UOR/Semester7/Advanced AI/Project/model"

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_dir)

# Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

print("✅ Model and tokenizer loaded from Drive!")

✅ Model and tokenizer loaded from Drive!
